# Dataset Final — Features de course
Combine : matching (2017-2025) + Komoot surfaces + features GPX
Output : dataset_final.csv

In [1]:
import os
import numpy as np
import pandas as pd
import gpxpy
from scipy.signal import find_peaks
from tqdm import tqdm

BASE       = '/Users/arthurdeletang/Desktop/Stage M1/Code/data'
GPX_DIR    = os.path.join(BASE, 'gpx_files_2')
MATCH_DIR  = os.path.join(BASE, 'matching', 'all')
KOMOOT_CSV = os.path.join(MATCH_DIR, 'komoot_surface_all.csv')
OUTPUT_CSV = os.path.join(MATCH_DIR, 'dataset_final.csv')

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ── 1. Charger tous les fichiers matching ─────────────────────────────────────
dfs = []
for year in range(2016, 2026):
    path = os.path.join(MATCH_DIR, f'matching_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[df['statut'] == 'match']
        dfs.append(df)
        print(f'{year} : {len(df)} courses')

df_match = pd.concat(dfs, ignore_index=True)
df_match = df_match.drop_duplicates(subset='fichier_gpx')
print(f'Total unique : {len(df_match)} courses')
df_match.head(3)

2017 : 595 courses
2018 : 1040 courses
2019 : 1062 courses
2020 : 448 courses
2021 : 700 courses
2022 : 813 courses
2023 : 937 courses
2024 : 890 courses
2025 : 906 courses
Total unique : 7391 courses


,jour,mois,annee,stage,nom_pcs,categorie,fichier_gpx,fichier_pcs,score,statut
0,5,1,2017,result,nc-australia-itt,NC,2017 Australian Road National Championships - ...,05_January_2017__result__nc-australia-itt_2017...,74.0,match
1,6,1,2017,result,nc-new-zealand-itt,NC,2017 New Zealand Road National Championships -...,06_January_2017__result__nc-new-zealand-itt_20...,80.0,match
2,8,1,2017,result,nc-australia,NC,2017 Australian Road National Championships.gpx,08_January_2017__result__nc-australia_2017__NC...,86.0,match


In [3]:
# ── 2. Joindre avec Komoot surfaces ───────────────────────────────────────────
df_komoot = pd.read_csv(KOMOOT_CSV)
print(f'Komoot : {len(df_komoot)} lignes')
print(f'Colonnes surface : {[c for c in df_komoot.columns if c not in ["fichier_gpx", "route_url"]]}')

df = df_match.merge(df_komoot, on='fichier_gpx', how='left')
print(f'Apres merge Komoot : {len(df)} lignes')
print(f'Sans donnees Komoot : {df["route_url"].isna().sum()} courses')

Komoot : 7272 lignes
Colonnes surface : ['road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unknown_km', 'path_km', 'unpaved_km', 'singletrack_km', 'compacted_gravel_km', '_km', 'alpine_km', 'ferry_km', 'alpine_path_km']
Apres merge Komoot : 7391 lignes
Sans donnees Komoot : 119 courses


In [4]:
import re

def parse_gpx_safe(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    content = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#)', '&amp;', content)
    return gpxpy.parse(content)

def find_gpx_path(fname, gpx_dir):
    full_path = os.path.join(gpx_dir, fname)
    if os.path.exists(full_path):
        return full_path
    truncated = fname.split(' / ')[0].strip() + '.gpx'
    trunc_path = os.path.join(gpx_dir, truncated)
    if os.path.exists(trunc_path):
        return trunc_path
    return None

def get_number_peaks(series, prominence):
    store = find_peaks(series, prominence=prominence)
    try:
        if (len(store[0]) > 1) and (np.min(np.diff(store[0])) < prominence):
            return len(store[1]['prominences'][np.diff(np.insert(store[0], 0, 0)) > prominence])
        else:
            return len(store[1]['prominences'])
    except:
        return len(store[1]['prominences'])

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def gradient_last_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1, 0, -1):
        dist_cumul += haversine(lats[i-1], lons[i-1], lats[i], lons[i])
        if dist_cumul >= km:
            elev_start = elevations[i-1]
            elev_end   = elevations[-1]
            deniv = elev_end - elev_start
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1])
                     for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def gradient_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        if dist_cumul >= km:
            deniv = elevations[i+1] - elevations[0]
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def denivele_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    deniv = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        diff = elevations[i+1] - elevations[i]
        if diff > 0:
            deniv += diff
        if dist_cumul >= km:
            break
    return round(deniv, 0)

def extract_gpx_features(filepath):
    try:
        gpx = parse_gpx_safe(filepath)
        points = gpx.tracks[0].segments[0].points
        elevations = [p.elevation or 0 for p in points]
        lats = [p.latitude for p in points]
        lons = [p.longitude for p in points]

        dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1])
                   for i in range(len(lats)-1))

        elev_diff = np.diff(elevations)
        denivele_pos = float(np.sum(elev_diff[elev_diff > 0]))
        denivele_neg = float(np.abs(np.sum(elev_diff[elev_diff < 0])))
        max_elev = float(np.max(elevations))
        min_elev = float(np.min(elevations))

        elev_series = elevations + [0]
        n = len(elev_series)

        def loc_last(prominence):
            peaks = find_peaks(elev_series, prominence=prominence)[0]
            return float(np.max(peaks) / n) if len(peaks) > 0 else 0.0

        last_points = min(500, len(elevations))
        elev_last = elevations[-last_points:]
        deniv_last_5km = float(np.sum(d for d in np.diff(elev_last) if d > 0))

        return {
            'distance_gpx_km':    round(dist, 2),
            'denivele_pos':       round(denivele_pos, 0),
            'denivele_neg':       round(denivele_neg, 0),
            'altitude_max':       round(max_elev, 0),
            'altitude_min':       round(min_elev, 0),
            'n_cols_hc':  get_number_peaks(elev_series, 800),
            'n_cols_cat1': get_number_peaks(elev_series, 640) - get_number_peaks(elev_series, 800),
            'n_cols_cat2': get_number_peaks(elev_series, 320) - get_number_peaks(elev_series, 640),
            'n_cols_cat3': get_number_peaks(elev_series, 160) - get_number_peaks(elev_series, 320),
            'n_cols_cat4': get_number_peaks(elev_series, 80)  - get_number_peaks(elev_series, 160),
            'loc_last_col_cat4':  round(loc_last(80), 4),
            'loc_last_col_cat3':  round(loc_last(160), 4),
            'loc_last_col_cat2':  round(loc_last(320), 4),
            'loc_last_col_cat1':  round(loc_last(640), 4),
            'loc_last_col_hc':    round(loc_last(800), 4),
            'gradient_last_1km':  gradient_last_km(lats, lons, elevations, 1),
            'gradient_last_3km':  gradient_last_km(lats, lons, elevations, 3),
            'gradient_last_5km':  gradient_last_km(lats, lons, elevations, 5),
            'denivele_last_5km':  round(deniv_last_5km, 0),
            'gradient_first_50km': gradient_first_km(lats, lons, elevations, 50),
            'denivele_first_50km': denivele_first_km(lats, lons, elevations, 50),
        }
    except Exception as e:
        print(f'  Erreur {os.path.basename(filepath)} : {e}')
        return {}

print('Fonctions chargees.')

Fonctions chargees.


In [5]:
gpx_features = []
for fname in tqdm(df['fichier_gpx']):
    if pd.isna(fname):
        gpx_features.append({})
        continue
    path = find_gpx_path(str(fname), GPX_DIR)
    if path:
        gpx_features.append(extract_gpx_features(path))
    else:
        gpx_features.append({})

df_gpx = pd.DataFrame(gpx_features)
print(f'Features GPX extraites : {df_gpx.shape}')
df_gpx.head(3)

  0%|          | 0/7391 [00:00<?, ?it/s]/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_35529/774333787.py:100: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  deniv_last_5km = float(np.sum(d for d in np.diff(elev_last) if d > 0))
  1%|          | 84/7391 [00:26<38:37,  3.15it/s]


KeyboardInterrupt: 

In [ ]:
# ── 5. Assemblage final ───────────────────────────────────────────────────────
df_final = pd.concat([df.reset_index(drop=True), df_gpx.reset_index(drop=True)], axis=1)

cols_matching = ['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_gpx']
cols_komoot   = [c for c in df_komoot.columns if c not in ['fichier_gpx']]
cols_gpx      = list(df_gpx.columns)
cols_finales  = [c for c in cols_matching + cols_komoot + cols_gpx if c in df_final.columns]

df_final = df_final[cols_finales]
print(f'Shape final : {df_final.shape}')
print(f'Colonnes : {list(df_final.columns)}')
df_final.head(5)

Shape final : (7391, 45)
Colonnes : ['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_gpx', 'route_url', 'road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unknown_km', 'path_km', 'unpaved_km', 'singletrack_km', 'compacted_gravel_km', '_km', 'alpine_km', 'ferry_km', 'alpine_path_km', 'distance_gpx_km', 'denivele_pos', 'denivele_neg', 'altitude_max', 'altitude_min', 'n_cols_cat4', 'n_cols_cat3', 'n_cols_cat2', 'n_cols_cat1', 'n_cols_hc', 'loc_last_col_cat4', 'loc_last_col_cat3', 'loc_last_col_cat2', 'loc_last_col_cat1', 'loc_last_col_hc', 'gradient_last_1km', 'gradient_last_3km', 'gradient_last_5km', 'denivele_last_5km']


,jour,mois,annee,stage,nom_pcs,categorie,fichier_gpx,route_url,road_km,street_km,...,n_cols_hc,loc_last_col_cat4,loc_last_col_cat3,loc_last_col_cat2,loc_last_col_cat1,loc_last_col_hc,gradient_last_1km,gradient_last_3km,gradient_last_5km,denivele_last_5km
0,5,1,2017,result,nc-australia-itt,NC,2017 Australian Road National Championships - ...,https://www.komoot.com/tour/2969352015,40.8,NaN,...,0.0,0.9974,0.0000,0.0000,0.0,0.0,2.0,1.47,1.24,306.0
1,6,1,2017,result,nc-new-zealand-itt,NC,2017 New Zealand Road National Championships -...,https://www.komoot.com/tour/2969526992,38.0,NaN,...,0.0,0.0000,0.0000,0.0000,0.0,0.0,0.5,-1.63,-0.22,178.0
2,8,1,2017,result,nc-australia,NC,2017 Australian Road National Championships.gpx,https://www.komoot.com/tour/2969352765,144.0,NaN,...,0.0,0.9700,0.0000,0.0000,0.0,0.0,-4.5,-2.83,-2.64,338.0
3,8,1,2017,result,nc-new-zealand,NC,2017 New Zealand Road National Championships.gpx,https://www.komoot.com/tour/2969527873,106.0,6.010,...,0.0,0.2077,0.2077,0.0000,0.0,0.0,-0.3,0.40,0.20,123.0
4,17,1,2017,stage_1,tour-down-under,2.UWT,2017 Santos Tour Down Under Stage 1.gpx,https://www.komoot.com/tour/2969570889,56.0,0.122,...,0.0,0.9642,0.3527,0.3527,0.0,0.0,-0.9,-1.20,-1.22,197.0


In [ ]:
# ── 6. Sauvegarde ─────────────────────────────────────────────────────────────
df_final.to_csv(OUTPUT_CSV, index=False)
print(f'Sauvegarde : {OUTPUT_CSV}')
print(f'Shape : {df_final.shape}')
print(f'\nCourses avec Komoot : {df_final["route_url"].notna().sum()}')
print(f'Courses avec GPX    : {df_final["distance_gpx_km"].notna().sum()}')
print(f'\nNaN par colonne :')
print(df_final.isna().sum()[df_final.isna().sum() > 0])

Sauvegarde : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/dataset_final.csv
Shape : (7391, 45)

Courses avec Komoot : 7272
Courses avec GPX    : 7225

NaN par colonne :
route_url                 119
road_km                   349
street_km                1137
state_road_km            1030
cycleway_km              3185
off-grid_(unknown)_km    3417
access_road_km           6301
asphalt_km                184
paved_km                  914
cobblestones_km          4530
unknown_km               2277
path_km                  3658
unpaved_km               4830
singletrack_km           5974
compacted_gravel_km      6803
_km                      7391
alpine_km                7376
ferry_km                 7389
alpine_path_km           7388
distance_gpx_km           166
denivele_pos              166
denivele_neg              166
altitude_max              166
altitude_min              166
n_cols_cat4               166
n_cols_cat3               166
n_cols_cat2               166
n_co

In [ ]:
print(df_final['stage'].value_counts().head(20))

stage
result      1859
stage_1     1011
stage_2     1004
stage_3      939
stage_4      777
stage_5      572
stage_6      292
stage_7      222
stage_8      151
prologue     112
stage_9       82
stage_10      68
stage_11      30
stage_12      30
stage_13      30
stage_14      27
stage_15      27
stage_16      27
stage_17      27
stage_18      27
Name: count, dtype: int64


In [ ]:
import pandas as pd
import os

OUTPUT_BASE = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

df = pd.read_csv(os.path.join(OUTPUT_BASE, 'all', 'dataset_final.csv'))

sans_komoot = df[df['route_url'].isna()][['fichier_gpx', 'annee', 'categorie']].dropna(subset=['fichier_gpx'])
print(f'Courses sans Komoot : {len(sans_komoot)}')
print(sans_komoot['annee'].value_counts().sort_index())
sans_komoot.head(119)

Courses sans Komoot : 119
annee
2017    12
2018    24
2019    18
2020     4
2021     7
2022     9
2023    20
2024    13
2025    12
Name: count, dtype: int64


,fichier_gpx,annee,categorie
90,2017 Colombian Road National Championships.gpx,2017,NC
143,2017 Volta Ciclista a Catalunya Stage 7.gpx,2017,2.UWT
157,2017 Tour of Thailand Stage 2.gpx,2017,2.1
219,2017 Vuelta Ciclista a la Comunidad de Madrid ...,2017,2.1
272,2017 Tour of Estonia Stage 2.gpx,2017,2.1
...,...,...,...
7148,2025 Tour Bitwa Warszawska Stage 1.gpx,2025,2.2
7217,2025 Maryland Cycling Classic.gpx,2025,1.Pro
7228,2025 Tour of Istanbul Stage 3.gpx,2025,2.1
7260,2025 Grand Prix Cycliste de Montreal.gpx,2025,1.UWT


In [ ]:
cols_komoot = [c for c in df_komoot.columns if c not in ['fichier_gpx', 'route_url', '_km']]
mask_avec_url = df['route_url'].notna()
df.loc[mask_avec_url, cols_komoot] = df.loc[mask_avec_url, cols_komoot].fillna(0)
print(f'NaN Komoot corriges pour {mask_avec_url.sum()} courses avec URL')

NaN Komoot corriges pour 7272 courses avec URL


In [ ]:
sans_komoot = df[df['route_url'].isna()][['fichier_gpx', 'annee', 'categorie', 'nom_pcs']]
print(f'Courses sans Komoot : {len(sans_komoot)}')
print(sans_komoot.to_string(index=False))

Courses sans Komoot : 119
                                                            fichier_gpx  annee categorie                                            nom_pcs
                         2017 Colombian Road National Championships.gpx   2017        NC                                        nc-colombia
                            2017 Volta Ciclista a Catalunya Stage 7.gpx   2017     2.UWT                                  volta-a-catalunya
                                      2017 Tour of Thailand Stage 2.gpx   2017       2.1                                   tour-of-thailand
              2017 Vuelta Ciclista a la Comunidad de Madrid Stage 3.gpx   2017       2.1      vuelta-international-a-la-cominudad-de-madrid
                                       2017 Tour of Estonia Stage 2.gpx   2017       2.1                                    tour-of-estonia
                                    2017 Koga Slag om Norg Rest day.gpx   2017       1.1                                       slag-om

In [ ]:
def gradient_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        if dist_cumul >= km:
            deniv = elevations[i+1] - elevations[0]
            return round(deniv / (km * 1000) * 100, 2)
    total_dist = sum(haversine(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    if total_dist == 0:
        return 0.0
    return round((elevations[-1] - elevations[0]) / (total_dist * 1000) * 100, 2)

def denivele_first_km(lats, lons, elevations, km):
    dist_cumul = 0.0
    deniv = 0.0
    for i in range(len(lats)-1):
        dist_cumul += haversine(lats[i], lons[i], lats[i+1], lons[i+1])
        diff = elevations[i+1] - elevations[i]
        if diff > 0:
            deniv += diff
        if dist_cumul >= km:
            break
    return round(deniv, 0)